# Part 6 · Notebook 04 — Implied volatility and the implied forward

**Sessions:** S4 (Implied volatility) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Solve for implied volatility with Newton's method.
2. See where Newton fails, and fall back to a bracketing solver.
3. Recover the rate and the forward from put–call parity.
4. Build the smile from out-of-the-money options with the right inputs.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

## 1. Newton's method

Implied vol is the σ that makes the model price equal the market price. Newton: `σ ← σ − (model(σ) − price) / vega(σ)`. Start at `sigma0`, stop when `|model − price| < tol`, and give up (NaN) if vega is tiny, σ leaves `(1e-4, 5)`, or `steps` run out.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def newton_iv(price, S, K, T, r, q, cp, sigma0=0.3, steps=20, tol=1e-10):
    sigma = sigma0
    for _ in range(steps):
        diff = p.bsm_price(S, K, T, r, q, sigma, cp) - price
        if abs(diff) < tol:
            return float(sigma)
        vega = p.greeks(S, K, T, r, q, sigma, cp)["vega"]
        if vega < 1e-8:
            return np.nan
        sigma = ...                               # ✍️ the Newton step
        if not 1e-4 < sigma < 5:
            return np.nan
    return np.nan

T, r, q = 30 / 365, 0.045, 0.013
cases = [(p.bsm_price(600.0, K, T, r, q, 0.22, cp), 600.0, K, T, r, q, cp) for K, cp in [(600.0, 1), (560.0, -1), (650.0, 1)]]
mine = [p.attempt(newton_iv, *c) for c in cases]
mine = p.check("newton_iv", mine, [p.newton_iv(*c) for c in cases])
mine

## 2. Where Newton breaks

Far out of the money and close to expiry, vega at the starting guess is almost zero and the first step shoots off. The fix is a **bracketing** method (Brent) that can't leave the interval, used when Newton fails. And before any of it, a price outside the no-arbitrage bounds has **no** implied vol: return NaN instead of a made-up number.

In [ ]:
rows = []
for K, dte, true in [(700.0, 7, 0.35), (760.0, 30, 0.45), (520.0, 3, 0.60), (600.0, 30, 0.18)]:
    Tk = dte / 365
    price = p.bsm_price(600.0, K, Tk, r, q, true, 1 if K >= 600 else -1)
    cp = 1 if K >= 600 else -1
    rows.append({"strike": K, "DTE": dte, "price": round(price, 6), "true σ": true,
                 "Newton from 0.3": p.newton_iv(price, 600.0, K, Tk, r, q, cp),
                 "Newton + Brent": p.implied_vol(price, 600.0, K, Tk, r, q, cp)})
display(pd.DataFrame(rows))
print("a price below intrinsic has no IV:", p.implied_vol(1.0, 600.0, 580.0, T, r, q, 1))

## 3. Rate and forward from put–call parity

Your IVs are only as good as `r`, `q` and the forward you feed in, and dividends are guesses. Parity gives them for free: `C − P = DF·F − DF·K`, a straight line in `K`. Fit `C − P = a + b·K` (`np.polyfit(K, C − P, 1)` returns `b, a`); then `DF = −b`, `r = −ln(DF)/T`, `F = a/DF`. Use mids of liquid near-the-money strikes.

In [ ]:
S = 600.0
chain = p.synthetic_chain(S, T, r, q)
liq = p.liquidity_filter(chain)
both = liq.pivot_table(index="strike", columns="cp", values="mid").dropna()
near = both[(both.index > 570) & (both.index < 630)]
near.head()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def implied_rate_forward(strikes, calls, puts, T):
    b, a = np.polyfit(np.asarray(strikes, float), np.asarray(calls, float) - np.asarray(puts, float), 1)
    df_ = ...                                     # ✍️ the discount factor
    return ..., ...                               # ✍️ (r, F)

mine = p.attempt(implied_rate_forward, near.index, near[1], near[-1], T)
mine = p.check("implied_rate_forward", mine, p.implied_rate_forward(near.index, near[1], near[-1], T))
print(f"implied r = {mine[0]:.3%} (true 4.500%), implied F = {mine[1]:.2f} (true {p.fair_value(S, r, q, T):.2f})")

## 4. The smile, done right

Build the smile from **out-of-the-money** options: puts below the forward, calls at or above it. In-the-money options are mostly intrinsic value, so their IV is ill-conditioned (a penny moves it a lot). Price them off the forward with Black-76. Compare with a common shortcut: guess `r = 4%`, ignore the dividend, use spot.

In [ ]:
r_imp, F_imp = mine
otm = liq[((liq.cp == -1) & (liq.strike < F_imp)) | ((liq.cp == 1) & (liq.strike >= F_imp))]
good = [p.implied_vol(m, F_imp, k, T, r_imp, r_imp, cp) for m, k, cp in zip(otm["mid"], otm.strike, otm.cp)]
guess = [p.implied_vol(m, S, k, T, 0.04, 0.0, cp) for m, k, cp in zip(liq["mid"], liq.strike, liq.cp)]
fig, ax = plt.subplots()
ax.plot(otm.strike, otm.iv_true * 100, color="black", lw=1, label="true smile")
ax.plot(otm.strike, np.array(good) * 100, "o", ms=4, label="OTM mids, implied r and F")
ax.plot(liq.strike[liq.cp == 1], np.array(guess)[liq.cp.to_numpy() == 1] * 100, "x", ms=4, label="calls, guessed r, spot, q = 0")
ax.plot(liq.strike[liq.cp == -1], np.array(guess)[liq.cp.to_numpy() == -1] * 100, "+", ms=5, label="puts, guessed r, spot, q = 0")
ax.set(xlabel="strike", ylabel="IV, %", title="Same quotes, different inputs"); ax.legend(); plt.show()
err = np.nanmax(np.abs(np.array(good) - otm.iv_true.to_numpy())) * 100
print(f"largest error with implied inputs: {err:.2f} vol points (from bid/ask rounding of the mids)")

With guessed inputs, calls and puts at the same strike disagree, and neither matches the truth: the gap is the wrong forward. With parity-implied inputs the OTM smile lands on the true one.

## Wrap-up

* Newton for speed, Brent as the fallback, NaN outside the bounds.
* Take `r` and the forward from parity, per expiry; read the smile from OTM options.
* Graded versions: `labs/part06/week21_pricing_iv` (IV with fallback statistics, vectorized chain IV) and Clinic W1 (our IV vs the broker's).